In [8]:

#==============================Bài1============================
import heapq

#  NODE
class Node:
    def __init__(self, mat, empty_pos, g, h, parent):
        self.mat = mat
        self.empty_pos = empty_pos
        self.g = g
        self.h = h
        self.parent = parent

    def __lt__(self, other):
        return (self.g + self.h) < (other.g + other.h)


# HEURISTIC
def manhattan(mat, goal_pos, n):
    dist = 0
    for i in range(n):
        for j in range(n):
            val = mat[i][j]
            if val != 0:
                x, y = goal_pos[val]
                dist += abs(i - x) + abs(j - y)
    return dist


#  UTILS
def find_zero(mat, n):
    for i in range(n):
        for j in range(n):
            if mat[i][j] == 0:
                return (i, j)


def mat_to_tuple(mat):
    return tuple(tuple(row) for row in mat)


def print_matrix(mat):
    for row in mat:
        print(*row)
    print()


def print_path(node):
    if node is None:
        return
    print_path(node.parent)
    print_matrix(node.mat)


# SOLVE
def solve(start, goal):
    n = len(start)

    # bảng vị trí đích (O(1))
    goal_pos = {}
    for i in range(n):
        for j in range(n):
            goal_pos[goal[i][j]] = (i, j)

    empty_pos = find_zero(start, n)

    dx = [1, 0, -1, 0]
    dy = [0, -1, 0, 1]

    pq = []

    # dùng dict thay vì set (quan trọng)
    visited = {}

    root = Node(start, empty_pos, 0, manhattan(start, goal_pos, n), None)
    heapq.heappush(pq, root)

    while pq:
        cur = heapq.heappop(pq)

        state = mat_to_tuple(cur.mat)

        #  nếu đã có đường tốt hơn → bỏ qua
        if state in visited and visited[state] <= cur.g:
            continue
        visited[state] = cur.g

        # đạt đích
        if cur.h == 0:
            print("Lời giải:")
            print_path(cur)
            print("Số bước:", cur.g)
            return

        x, y = cur.empty_pos

        for i in range(4):
            nx = x + dx[i]
            ny = y + dy[i]

            if 0 <= nx < n and 0 <= ny < n:
                #  copy nhanh hơn deepcopy
                new_mat = [row[:] for row in cur.mat]

                new_mat[x][y], new_mat[nx][ny] = new_mat[nx][ny], new_mat[x][y]

                new_state = mat_to_tuple(new_mat)

                # cắt nhánh mạnh
                if new_state in visited and visited[new_state] <= cur.g + 1:
                    continue

                child = Node(
                    new_mat,
                    (nx, ny),
                    cur.g + 1,
                    manhattan(new_mat, goal_pos, n),
                    cur
                )

                heapq.heappush(pq, child)

    print(" Không tìm thấy lời giải")
    #  TEST
if __name__ == "__main__":
    print("===== Bài1 =====")
    start1 = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9,10,11, 0],
        [13,14,15,12]
    ]

    goal = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9,10,11,12],
        [13,14,15, 0]
    ]

    solve(start1, goal)

#===============================Bài2================================
import heapq

# NODE
class Node:
    def __init__(self, city, visited, g, h, path):
        self.city = city
        self.visited = visited
        self.g = g  # chi phí đã đi
        self.h = h  # heuristic
        self.path = path

    def __lt__(self, other):
        return (self.g + self.h) < (other.g + other.h)


# HEURISTIC
def heuristic(graph, visited):
    # lấy cạnh nhỏ nhất chưa dùng (đơn giản nhưng hiệu quả hơn h=0)
    min_edge = float('inf')
    n = len(graph)

    for i in range(n):
        if i not in visited:
            for j in range(n):
                if i != j:
                    min_edge = min(min_edge, graph[i][j])

    return min_edge if min_edge != float('inf') else 0

# A*
def a_star_tsp(graph, start):
    n = len(graph)

    pq = []
    start_node = Node(start, {start}, 0, 0, [start])
    heapq.heappush(pq, start_node)

    while pq:
        node = heapq.heappop(pq)

        # nếu đi hết các thành phố
        if len(node.visited) == n:
            total_cost = node.g + graph[node.city][start]
            path = node.path + [start]

            print(" Đường đi tối ưu:")
            print(" → ".join(map(str, path)))
            print(" Chi phí:", total_cost)
            return path, total_cost

        # mở rộng
        for next_city in range(n):
            if next_city not in node.visited:
                new_visited = node.visited.copy()
                new_visited.add(next_city)

                g = node.g + graph[node.city][next_city]
                h = heuristic(graph, new_visited)

                new_node = Node(
                    next_city,
                    new_visited,
                    g,
                    h,
                    node.path + [next_city]
                )

                heapq.heappush(pq, new_node)

    print(" Không tìm được lời giải")
    return None


# TEST
if __name__ == "__main__":
    print("===== Bài2:BÀI TOÁN NGƯỜI GIAO HÀNG (TSP - A*) =====")
    graph = [
        [0, 10, 15, 20],
        [10, 0, 35, 25],
        [15, 35, 0, 30],
        [20, 25, 30, 0]
    ]

    a_star_tsp(graph, 0)

===== Bài1 =====
Lời giải:
1 2 3 4
5 6 7 8
9 10 11 0
13 14 15 12

1 2 3 4
5 6 7 8
9 10 11 12
13 14 15 0

Số bước: 1
===== Bài2:BÀI TOÁN NGƯỜI GIAO HÀNG (TSP - A*) =====
 Đường đi tối ưu:
0 → 1 → 3 → 2 → 0
 Chi phí: 80
